# Clase 104 — Callbacks, TensorBoard y guardar/restaurar modelos

Los **callbacks** inyectan lógica en el loop de entrenamiento sin modificarlo:
early stopping, checkpoints, ajuste de learning rate, logging a TensorBoard. Además
vemos cómo **guardar y restaurar** correctamente (arquitectura + pesos + optimizador).

Requiere: `tensorflow` / `keras`, `tensorboard`. Se ejecuta en Colab con GPU.

## 1. Datos y un modelo base reutilizable

In [ ]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(42)

(X_tr, y_tr), (X_te, y_te) = keras.datasets.fashion_mnist.load_data()
X_tr = X_tr.astype("float32") / 255.0
X_te = X_te.astype("float32") / 255.0

def build_model():
    m = keras.Sequential([
        keras.Input(shape=(28, 28)),
        layers.Flatten(),
        layers.Dense(300, activation="relu"),
        layers.Dense(100, activation="relu"),
        layers.Dense(10, activation="softmax"),
    ])
    m.compile(optimizer="adam", loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
    return m

print("modelo base:", build_model().count_params(), "parámetros")

## 2. `EarlyStopping` + `ModelCheckpoint`

In [ ]:
early = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=10, restore_best_weights=True)
checkpoint = keras.callbacks.ModelCheckpoint(
    "best.keras", monitor="val_accuracy", mode="max", save_best_only=True)

modelo = build_model()
hist = modelo.fit(X_tr, y_tr, validation_split=0.1,
                  epochs=100, batch_size=128,
                  callbacks=[early, checkpoint], verbose=0)
print(f"entrenó {len(hist.history['loss'])} épocas (EarlyStopping cortó antes de 100)")
print("best.keras contiene el modelo con mayor val_accuracy")

## 3. `ReduceLROnPlateau`: bajar el LR cuando se estanca

In [ ]:
reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=0)

modelo2 = build_model()
h2 = modelo2.fit(X_tr, y_tr, validation_split=0.1,
                 epochs=15, batch_size=128,
                 callbacks=[reduce_lr], verbose=0)
# Keras registra 'learning_rate' en el history cuando el LR cambia
lrs = h2.history.get("learning_rate", [])
print("learning rates por época:", [round(float(v), 6) for v in lrs][:15])

## 4. TensorBoard para visualizar el entrenamiento

In [ ]:
import time
log_dir = f"./logs/run-{int(time.time())}"
tensorboard_cb = keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

modelo3 = build_model()
modelo3.fit(X_tr, y_tr, validation_split=0.1,
            epochs=5, batch_size=128,
            callbacks=[tensorboard_cb], verbose=0)
print(f"logs escritos en: {log_dir}")
print("Abrir con:  tensorboard --logdir=./logs")
print("En Colab:   %load_ext tensorboard  y  %tensorboard --logdir=./logs")

## 5. Callback custom: loggear a CSV con `on_epoch_end`

In [ ]:
import csv

class CSVLoggerSimple(keras.callbacks.Callback):
    def __init__(self, ruta):
        super().__init__()
        self.ruta = ruta
        self.filas = []

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        # self.model está disponible dentro del callback
        lr = float(self.model.optimizer.learning_rate.numpy())
        self.filas.append({
            "epoch": epoch,
            "loss": logs.get("loss"),
            "val_loss": logs.get("val_loss"),
            "lr": lr,
        })

    def on_train_end(self, logs=None):
        with open(self.ruta, "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=["epoch", "loss", "val_loss", "lr"])
            w.writeheader(); w.writerows(self.filas)

modelo4 = build_model()
modelo4.fit(X_tr, y_tr, validation_split=0.1, epochs=3,
            batch_size=128, callbacks=[CSVLoggerSimple("historia.csv")], verbose=0)
print("métricas por época guardadas en historia.csv")

## 6. Guardar, restaurar y continuar entrenamiento

In [ ]:
# Guardado completo: arquitectura + pesos + estado del optimizador (momentum Adam)
modelo.save("modelo_completo.keras")
restaurado = keras.models.load_model("modelo_completo.keras")

acc_original = modelo.evaluate(X_te, y_te, verbose=0)[1]
acc_restaurado = restaurado.evaluate(X_te, y_te, verbose=0)[1]
print(f"accuracy original:   {acc_original:.4f}")
print(f"accuracy restaurado: {acc_restaurado:.4f}  (idéntico)")

# continuar entrenando desde donde quedó, sin perder el estado del optimizador
restaurado.fit(X_tr, y_tr, epochs=2, batch_size=128, verbose=0)
print("entrenamiento reanudado desde el checkpoint.")

## Ejercicios

1. **EarlyStopping + Checkpoint**: verificá que `best.keras` corresponde al epoch de
   mayor `val_accuracy`, no al último.
2. **ReduceLROnPlateau**: graficá el learning rate a lo largo de las épocas y
   marcá dónde se redujo.
3. **Custom callback**: extendé `CSVLoggerSimple` para que también registre la
   accuracy y el tiempo por época.
4. **Restaurar y continuar**: entrená 10 épocas, guardá, recargá y continuá 5 más;
   comprobá que el momentum de Adam se preservó (la loss no salta al reanudar).

## Conclusiones

- Los **callbacks** añaden comportamiento al training vía hooks (`on_epoch_end`, etc.).
- `EarlyStopping(restore_best_weights=True)` + `ModelCheckpoint(save_best_only=True)` son el dúo estándar.
- `ReduceLROnPlateau` baja el LR de forma reactiva cuando la métrica se estanca.
- **TensorBoard** visualiza scalars, histogramas y embeddings; se sirve con `tensorboard --logdir`.
- El formato **`.keras`** guarda todo (incluido el optimizador), permitiendo reanudar el entrenamiento sin pérdida.